# 01a: EyePACS probe

Short check before the full preprocessing run (`01_preprocess_eyepacs`). It answers:

1. How does Kaggle present the EyePACS images: extracted `.jpeg` files or the split `train.zip.001-005`?
2. Do the labelled images in `trainLabels.csv` all exist?
3. Does the RetinaGuard preprocessing (crop to fundus, 512 px) work on real images, how fast, and how big is the output?

**Settings** (right-hand panel): Accelerator **None** (CPU is enough), Internet **On**.
**Input:** *Add Input* → *Competitions* → **Diabetic Retinopathy Detection**.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
N_SAMPLE = 200  # images to preprocess in this probe
SEED = 0

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Locate the competition files ------------------------------------------
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
candidates = [
    p
    for pattern in ("*/trainLabels.csv*", "*/*/trainLabels.csv*", "*/*/*/trainLabels.csv*")
    for p in INPUT_ROOT.glob(pattern)
]
assert candidates, "trainLabels.csv not found: is the competition attached under Add Input?"
labels_path = candidates[0]
DATA_DIR = labels_path.parent
print("data dir:", DATA_DIR)
for p in sorted(DATA_DIR.iterdir()):
    size = p.stat().st_size / 1e9 if p.is_file() else float("nan")
    print(f"  {p.name:<28} {'dir' if p.is_dir() else f'{size:6.2f} GB'}")

In [ ]:
# --- 2. Open the images, whichever layout Kaggle uses -----------------------
import time

from retinaguard.data.archive import iter_members, open_split_zip
from retinaguard.data.sources import eyepacs

t0 = time.perf_counter()
jpeg_dir = next((d for d in (DATA_DIR / "train", DATA_DIR / "train" / "train") if d.is_dir()), None)
first_part = DATA_DIR / "train.zip.001"
if jpeg_dir is not None:
    LAYOUT = "extracted"
    files = {p.stem: p for p in jpeg_dir.glob("*.jpeg")}

    def read_bytes(name):
        return files[name].read_bytes()
elif first_part.exists():
    LAYOUT = "split_zip"
    archive = open_split_zip(first_part)
    files = {Path(info.filename).stem: info for info in iter_members(archive, ".jpeg")}

    def read_bytes(name):
        return archive.read(files[name])
else:
    raise FileNotFoundError(f"no train/ folder or train.zip.001 in {DATA_DIR}")
print(f"layout: {LAYOUT}, {len(files):,} images indexed in {time.perf_counter() - t0:.1f}s")

labels = eyepacs.read_labels(labels_path)
names = set(labels["image_id"].str.removeprefix("eyepacs:"))
missing = sorted(names - files.keys())
unlabelled = sorted(files.keys() - names)
print(f"labelled: {len(names):,}  missing images: {len(missing)}")
print(f"images without label: {len(unlabelled)}")
print("first missing:", missing[:5])

In [ ]:
# --- 3. Preprocess a random sample --------------------------------------------
import random

import pandas as pd

from retinaguard.data.preprocess import decode_rgb, encode_jpeg, preprocess

sample = random.Random(SEED).sample(sorted(names & files.keys()), N_SAMPLE)
rows, failures, examples = [], [], {}
for name in sample:
    t = time.perf_counter()
    try:
        raw = decode_rgb(read_bytes(name))
        out, circle = preprocess(raw)
        jpeg = encode_jpeg(out)
    except Exception as exc:  # recorded, not hidden: failures are part of the probe result
        failures.append({"name": name, "error": repr(exc)})
        continue
    rows.append(
        {
            "name": name,
            "width": raw.shape[1],
            "height": raw.shape[0],
            "radius": circle.radius,
            "radius_ratio": 2 * circle.radius / raw.shape[1],
            "jpeg_kb": len(jpeg) / 1024,
            "seconds": time.perf_counter() - t,
        }
    )
    if len(examples) < 6:
        examples[name] = (raw, out)

stats = pd.DataFrame(rows)
n_total = len(names)
summary = {
    "layout": LAYOUT,
    "images_indexed": len(files),
    "labelled": n_total,
    "missing_images": len(missing),
    "sample": N_SAMPLE,
    "failures": len(failures),
    "median_seconds_per_image": round(float(stats["seconds"].median()), 3),
    "est_hours_single_process": round(float(stats["seconds"].mean()) * n_total / 3600, 2),
    "mean_output_kb": round(float(stats["jpeg_kb"].mean()), 1),
    "est_output_gb": round(float(stats["jpeg_kb"].mean()) * n_total / 1024**2, 2),
    "raw_width_min_max": [int(stats["width"].min()), int(stats["width"].max())],
    "raw_height_min_max": [int(stats["height"].min()), int(stats["height"].max())],
    "radius_ratio_min_median_max": [
        round(float(q), 3) for q in stats["radius_ratio"].quantile([0, 0.5, 1])
    ],
}
print(pd.Series(summary).to_string())
print(failures[:10])

In [ ]:
# --- 4. Look at a few results (raw | cropped 512 | normalised) ----------------
import matplotlib.pyplot as plt

from retinaguard.data.preprocess import local_mean_normalise

worst = stats.nsmallest(3, "radius_ratio")["name"].tolist()  # most suspicious crops
for name in worst:
    if name not in examples:
        raw = decode_rgb(read_bytes(name))
        examples[name] = (raw, preprocess(raw)[0])

fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
titles = ("raw", "crop 512", "normalised")
for row, (name, (raw, out)) in zip(axes, examples.items(), strict=True):
    for ax, img, title in zip(row, (raw, out, local_mean_normalise(out)), titles, strict=True):
        ax.imshow(img)
        ax.set_title(f"{name} {title}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- 5. Save the probe results -------------------------------------------------
import json

out_dir = Path("/kaggle/working/probe")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
stats.to_csv(out_dir / "sample_stats.csv", index=False)
pd.DataFrame(failures).to_csv(out_dir / "failures.csv", index=False)
fig.savefig(out_dir / "examples.png", dpi=80)
print((out_dir / "summary.json").read_text())

**Send back:** the printed summary from step 3 (or `probe/summary.json`), any failures, and whether the crops in step 4 look right (fundus centred and filling the square, nothing cut off).